# PHASE: W9-PRE

**SCIENTIFIC QUESTION:** Which official textual generator can use the same FIT evidence within <20 GB peak allocated VRAM?

**PREREQUISITES:** Official local caches/access; first-pass SARA remains canonical Mistral. No alternative receives SARA latent vectors.

**INPUT ARTIFACTS:** `configs/week9/w9pre_models.yaml`, fixed FIT subset and shared textual evidence.

**EXPECTED OUTPUTS:** cache manifest, private raw outputs, public final-answer results, measured precision/VRAM/cost and optional FIT-frozen panel.

**CURRENT SPLIT:** FIT only. TEST and confirmation data are unavailable here.

**FULL RUN FLAG:** `RUN_FULL=False`. All stages require explicit flags. Screen is a manual 32–64-query paper-grouped FIT experiment. Run All defaults to inspection only. Raw reasoning is private and must not enter publication artifacts.

In [1]:
from pathlib import Path
import sys,json,subprocess
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/week9').is_dir())
RUN_FULL = False
RUN_SMOKE = False
RUN_CACHE = False
FREEZE_PANEL = False
SMOKE_LIMIT = 2
MODELS = ['M0','M2']  # Edit after inspecting cache/access/VRAM audit; no panel chosen here.
SCRIPT=ROOT/'scripts/week9/run_w9pre_models.py'
def run(stage,*extra):
    command=[str(ROOT/'sara_env/bin/python'),str(SCRIPT),'--stage',stage]
    for model in MODELS:command.extend(['--model',model])
    subprocess.run(command+list(extra),cwd=ROOT,check=True)


In [2]:
if RUN_CACHE:run('cache')
if RUN_SMOKE:run('smoke','--smoke-limit',str(SMOKE_LIMIT),'--resume')
if RUN_FULL:
    run('screen','--resume',*(['--freeze'] if FREEZE_PANEL else []))
else:
    print('FIT screen disabled. No model is selected by a smoke test.')


FIT screen disabled. No model is selected by a smoke test.


In [3]:
folder=ROOT/'results/week9/preflight'
for name in ['model_cache_manifest.json','smoke_status.json','smoke_summary.json','screen_summary.json']:
    path=folder/name
    if path.exists():
        value=json.loads(path.read_text());print(name)
        display(pd.DataFrame(value['models'] if isinstance(value,dict) else value))
    else:print(name,': pending; no fallback results')


model_cache_manifest.json


,model_id,revision,local_cache_path,download_status,disk_size,license,gated,access_status,error,official_weight_bytes,free_disk_before
0,Qwen/Qwen2.5-7B-Instruct,a09a35458c702b33eeacc393d103063234e8bc28,/home/gunavenkat/.cache/huggingface/hub/models...,COMPLETE,1.523127e+10,Apache-2.0,False,existing official snapshot verified,NaN,NaN,NaN
1,mistralai/Mistral-7B-Instruct-v0.2,63a8b081895390a26e140280378bc85ec8bce07a,/home/gunavenkat/.cache/huggingface/hub/models...,COMPLETE,1.448350e+10,Apache-2.0,False,existing official snapshot verified,NaN,NaN,NaN
2,meta-llama/Llama-3.1-8B-Instruct,NaN,NaN,ACCESS_OR_NETWORK_ERROR,NaN,Llama-3.1,True,GatedRepoError,401 Client Error. (Request ID: Root=1-6ad7a8d1...,NaN,NaN
3,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,916b56a44061fd5cd7d6a8fb632557ed4f724f60,/home/gunavenkat/.cache/huggingface/hub/models...,COMPLETE,1.523127e+10,MIT (upstream Qwen Apache-2.0),False,official snapshot verified,NaN,1.523127e+10,3.801140e+10
4,Qwen/Qwen3-8B,b968826d9c46dd6066d109eabc6255188de91218,NaN,INSUFFICIENT_DISK,NaN,Apache-2.0,False,metadata accessible,NaN,1.638152e+10,2.277077e+10
5,google/gemma-3-12b-it,NaN,NaN,ACCESS_OR_NETWORK_ERROR,NaN,Gemma terms,True,GatedRepoError,401 Client Error. (Request ID: Root=1-6ad7a984...,NaN,NaN


smoke_status.json


,model_id,status,precision,peak_memory_allocated,peak_memory_reserved,dtype,quantization,prompt_tokens,generated_tokens,reason
0,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,FAIL,fp16,15342163968,1.556716e+10,float16,none,NaN,NaN,RuntimeError: Numerically invalid generation s...
1,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,FAIL,int8,8941492224,1.448293e+10,float16,bitsandbytes-LLM.int8,NaN,NaN,RuntimeError: Numerically invalid generation s...
2,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,FAIL,nf4,5742134272,1.440534e+10,float16,bitsandbytes-NF4-double-quant,NaN,NaN,RuntimeError: Numerically invalid generation s...
3,mistralai/Mistral-7B-Instruct-v0.2,PASS,fp16,14631406592,NaN,NaN,NaN,NaN,NaN,NaN
4,Qwen/Qwen2.5-7B-Instruct,FAIL,fp16,15325345280,1.554199e+10,float16,none,NaN,NaN,RuntimeError: Numerically invalid generation s...
5,Qwen/Qwen2.5-7B-Instruct,PASS,int8,8962191360,NaN,NaN,NaN,NaN,NaN,NaN


smoke_summary.json


,model_id,revision,precision,task,queries,f1,exact_match,help_rate,hurt_rate,evidence_utilization_rate,...,mean_generated_tokens,mean_total_query_tokens,mean_latency,p50_latency,p95_latency,peak_memory_allocated,peak_memory_reserved,mean_input_characters,vram_status,quantization
0,mistralai/Mistral-7B-Instruct-v0.2,63a8b081895390a26e140280378bc85ec8bce07a,fp16,T0,2,0.029412,0.0,0.0,1.0,1.0,...,50.0,258.5,2.420182,2.420182,3.994411,14555941376,14852030464,950.0,PASS,none
1,mistralai/Mistral-7B-Instruct-v0.2,63a8b081895390a26e140280378bc85ec8bce07a,fp16,T1,2,NaN,NaN,NaN,NaN,NaN,...,256.0,492.5,8.922102,8.922102,8.931215,14631406592,14860419072,1055.0,PASS,none
2,mistralai/Mistral-7B-Instruct-v0.2,63a8b081895390a26e140280378bc85ec8bce07a,fp16,T2,2,NaN,NaN,NaN,NaN,NaN,...,256.0,1414.0,8.907345,8.907345,8.923459,14630382592,14860419072,1095.0,PASS,none
3,mistralai/Mistral-7B-Instruct-v0.2,63a8b081895390a26e140280378bc85ec8bce07a,fp16,T3,2,0.041667,0.0,0.0,1.0,1.0,...,32.0,1056.5,1.135585,1.135585,1.641224,14522857984,14860419072,508.0,PASS,none
4,mistralai/Mistral-7B-Instruct-v0.2,63a8b081895390a26e140280378bc85ec8bce07a,fp16,T4,2,0.100000,0.0,0.0,1.0,1.0,...,29.5,1134.0,1.062292,1.062292,1.421605,14543546368,14860419072,916.0,PASS,none
5,Qwen/Qwen2.5-7B-Instruct,a09a35458c702b33eeacc393d103063234e8bc28,int8,T0,2,0.100000,0.0,0.0,1.0,1.0,...,14.5,230.0,1.422038,1.422038,1.548704,8962191360,14300479488,1079.0,PASS,bitsandbytes-LLM.int8
6,Qwen/Qwen2.5-7B-Instruct,a09a35458c702b33eeacc393d103063234e8bc28,int8,T1,2,NaN,NaN,NaN,NaN,NaN,...,69.0,302.5,6.499345,6.499345,6.844739,8962191360,14306770944,1184.0,PASS,bitsandbytes-LLM.int8
7,Qwen/Qwen2.5-7B-Instruct,a09a35458c702b33eeacc393d103063234e8bc28,int8,T2,2,NaN,NaN,NaN,NaN,NaN,...,72.0,1225.5,6.785995,6.785995,6.880332,8962191360,14313062400,1224.0,PASS,bitsandbytes-LLM.int8
8,Qwen/Qwen2.5-7B-Instruct,a09a35458c702b33eeacc393d103063234e8bc28,int8,T3,2,0.055556,0.0,0.0,1.0,1.0,...,22.0,1054.0,2.099664,2.099664,2.872381,8962191360,14313062400,637.0,PASS,bitsandbytes-LLM.int8
9,Qwen/Qwen2.5-7B-Instruct,a09a35458c702b33eeacc393d103063234e8bc28,int8,T4,2,0.360714,0.0,0.0,1.0,1.0,...,30.5,1137.0,2.917480,2.917480,4.487207,8962191360,14313062400,1045.0,PASS,bitsandbytes-LLM.int8


screen_summary.json : pending; no fallback results


Inspect the fixed source/final-answer sample and provide `qualitative_review.json` with `approved_model_ids` and the hash of `textual_fit_evidence.json` before screen freezing. This prevents selection solely on F1 when answer entailment is unmeasured. Use the published rule in the runbook. F0/F1/F2 prompts and all model token/precision costs remain explicit. No full W9 pipeline is launched by this notebook.

Current feasibility does not imply a selected panel. M0 extraction-format failures require inspection; Qwen2.5 requires the measured int8 mode on this host. DeepSeek has a separate NF4/FP32 diagnostic configuration and lacks final answers under the small generation cap. Llama/Gemma access and Qwen3 disk capacity remain blocked. Never expand to confirmation data to choose a model.